# DJIA Four-Asset Library Walkthrough

This notebook is a compact end-to-end demo of the cleaned library API. It uses four DJIA names with a shared 2019 construction window and a 2020 out-of-sample evaluation window.


## What This Covers

- loading market data from a saved local Yahoo-style snapshot
- creating a `Universe` with explicit construction dates
- building multiple portfolio constructors on the same asset set
- inspecting in-sample metrics and HRP diagnostics
- running backtests and Monte Carlo evaluation
- plotting with `PortfolioVisualizer`
- saving the standard library output structure under `outputs/runs/`

This is meant as a library walkthrough rather than a thesis-results notebook, so it uses the standard reusable library surfaces instead of the larger thesis experiment runner.


In [1]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "portafolios").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Could not locate the repository root from the current working directory.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json

import pandas as pd
from IPython.display import display

from portafolios import (
    Universe,
    local_loader,
    EqualWeightConstructor,
    Markowitz,
    NaiveRiskParity,
    HRPStyle,
    HRPRecursive,
    Backtester,
    MonteCarloEngine,
    PortfolioVisualizer,
)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

PROJECT_ROOT


WindowsPath('C:/Users/narro/Desktop/semestre/honores_actuaria')

In [2]:
DATA_PATH = PROJECT_ROOT / "outputs" / "final_experimental_setup" / "data" / "yfinance_final_setup_raw.csv"
TICKERS = ["AAPL", "MSFT", "JPM", "V"]

START = "2019-01-02"
END = "2020-12-31"
CONSTRUCTION_START = "2019-01-02"
CONSTRUCTION_END = "2019-12-31"
BACKTEST_START = "2020-01-02"
BACKTEST_END = "2020-12-31"

MC_SIMULATIONS = 300
MC_SEED = 42
MC_INITIAL_VALUE = 1.0
MAX_MC_PATHS_TO_SAVE = 60

RUN_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "runs"
UNIVERSE_NAME = "djia_four_asset_library_walkthrough"

config = pd.Series(
    {
        "data_path": str(DATA_PATH.relative_to(PROJECT_ROOT)),
        "tickers": TICKERS,
        "start": START,
        "end": END,
        "construction_start": CONSTRUCTION_START,
        "construction_end": CONSTRUCTION_END,
        "backtest_start": BACKTEST_START,
        "backtest_end": BACKTEST_END,
        "mc_simulations": MC_SIMULATIONS,
        "mc_seed": MC_SEED,
        "run_output_dir": str(RUN_OUTPUT_DIR.relative_to(PROJECT_ROOT)),
    },
    name="demo_config",
)

display(config)


data_path             outputs\final_experimental_setup\data\yfinance...
tickers                                            [AAPL, MSFT, JPM, V]
start                                                        2019-01-02
end                                                          2020-12-31
construction_start                                           2019-01-02
construction_end                                             2019-12-31
backtest_start                                               2020-01-02
backtest_end                                                 2020-12-31
mc_simulations                                                      300
mc_seed                                                              42
run_output_dir                                             outputs\runs
Name: demo_config, dtype: object

## Build The Universe

This uses the cleaned public API and a saved local snapshot, so the walkthrough is fully reproducible without a live download.


In [3]:
universe = Universe(
    universe_name=UNIVERSE_NAME,
    loader=local_loader,
    loader_kwargs={"path": DATA_PATH, "prefer_adj_close": True},
    tickers=TICKERS,
    start=START,
    end=END,
    construction_start=CONSTRUCTION_START,
    construction_end=CONSTRUCTION_END,
    base_output_dir=RUN_OUTPUT_DIR,
    auto_save_data=False,
).prepare_data()

run_dir = universe.output_dir
run_dir


WindowsPath('C:/Users/narro/Desktop/semestre/honores_actuaria/outputs/runs/djia_four_asset_library_walkthrough')

In [4]:
overview = pd.DataFrame(
    {
        "value": [
            universe.prices.index.min().date(),
            universe.prices.index.max().date(),
            len(universe.prices),
            len(universe.returns),
            list(universe.prices.columns),
            universe.construction_start.date(),
            universe.construction_end.date(),
            str(run_dir.relative_to(PROJECT_ROOT)),
        ]
    },
    index=[
        "price_start",
        "price_end",
        "n_price_rows",
        "n_return_rows",
        "tickers",
        "construction_start",
        "construction_end",
        "output_dir",
    ],
)

display(overview)
display(universe.prices.head())
display(universe.returns.head())


,value
price_start,2019-01-02
price_end,2020-12-31
n_price_rows,522
n_return_rows,521
tickers,"[AAPL, MSFT, JPM, V]"
construction_start,2019-01-02
construction_end,2019-12-31
output_dir,outputs\runs\djia_four_asset_library_walkthrough


,AAPL,MSFT,JPM,V
Date,,,,
2019-01-02,37.503731,94.397141,80.836479,126.329468
2019-01-03,33.768066,90.924469,79.687675,121.776978
2019-01-04,35.209618,95.153297,82.625404,127.023232
2019-01-07,35.131237,95.274643,82.682831,129.313721
2019-01-08,35.800961,95.965454,82.526924,130.017029


,AAPL,MSFT,JPM,V
Date,,,,
2019-01-03,-0.099608,-0.036788,-0.014211,-0.036037
2019-01-04,0.042690,0.046509,0.036866,0.043081
2019-01-07,-0.002226,0.001275,0.000695,0.018032
2019-01-08,0.019063,0.007251,-0.001886,0.005439
2019-01-09,0.016981,0.014300,-0.001690,0.011770


## Build Constructors On The Same Universe

This is the standard comparison workflow: all methods use the same assets and the same construction window.


In [5]:
universe.build(EqualWeightConstructor())
universe.build(Markowitz(), ret_kind="simple", allow_short=False, on_failure="use_initial_weights")
universe.build(NaiveRiskParity())
universe.build(HRPStyle(n_clusters=2, display_name="HRP Style (2 Clusters)"))
universe.build(HRPRecursive())

universe.list_constructions()


['equal_weight',
 'markowitz_max_sharpe',
 'naive_risk_parity',
 'hrp_style',
 'hrp_recursive']

In [6]:
construction_summary = pd.DataFrame(
    [
        {
            "name": name,
            "display_name": result.display_name,
            "method_id": result.method_id,
            "n_selected": result.metrics_insample.get("n_selected"),
            "expected_return": result.metrics_insample.get("expected_return"),
            "volatility": result.metrics_insample.get("volatility"),
            "sharpe_m": result.metrics_insample.get("sharpe_m"),
        }
        for name, result in universe.constructions.items()
    ]
).set_index("name")

weights_table = pd.DataFrame(
    {
        name: universe.get_construction(name).weights
        for name in universe.list_constructions()
    }
).fillna(0.0)

display(construction_summary.sort_index())
display(weights_table)


,display_name,method_id,n_selected,expected_return,volatility,sharpe_m
name,,,,,,
equal_weight,Equal Weight,equal_weight,4,0.001840,0.010394,0.176979
hrp_recursive,HRP Recursive,hrp_recursive,4,0.001711,0.009959,0.171808
hrp_style,HRP Style (2 Clusters),hrp_style,4,0.001725,0.010028,0.172007
markowitz_max_sharpe,Markowitz Max Sharpe,markowitz_max_sharpe,4,0.001962,0.010960,0.179052
naive_risk_parity,Naive Risk Parity (1/sigma),naive_risk_parity,4,0.001777,0.010137,0.175328


,equal_weight,markowitz_max_sharpe,naive_risk_parity,hrp_style,hrp_recursive
AAPL,0.25,0.336790,0.193357,0.166667,0.157967
JPM,0.25,0.295471,0.270954,0.500000,0.488927
MSFT,0.25,0.293864,0.255025,0.166667,0.159691
V,0.25,0.073875,0.280664,0.166667,0.193415


In [7]:
hrp_result = universe.get_construction("hrp_style")
hrp_diag = hrp_result.hrp_diagnostics

display(pd.Series(hrp_diag.cluster_assets, name="assets_in_cluster") if hrp_diag is not None else pd.Series(dtype=object))
display(universe.get_hrp_dist_matrix("hrp_style"))


C0    [AAPL, MSFT, V]
C1              [JPM]
Name: assets_in_cluster, dtype: object

,AAPL,MSFT,V,JPM
AAPL,0.000000,0.377807,0.460388,0.541875
MSFT,0.377807,0.000000,0.275876,0.543676
V,0.460388,0.275876,0.000000,0.594353
JPM,0.541875,0.543676,0.594353,0.000000


## Run Evaluation

For this walkthrough we pass explicit 2020 dates into `Backtester.run_all(...)`. If you omit them, `Backtester.run()` defaults to the first available date after `construction_end` through the last available date in the universe.


In [8]:
backtest_results = Backtester.run_all(
    universe,
    start_date=BACKTEST_START,
    end_date=BACKTEST_END,
    ann_factor=252,
    attach=True,
)

backtest_summary = pd.DataFrame(
    [
        {"name": name, **result.summary_metrics}
        for name, result in backtest_results.items()
    ]
).set_index("name")

display(backtest_summary.sort_values("sharpe_ratio", ascending=False))


,n_periods,total_return,annualized_return,annualized_volatility,sharpe_ratio,max_drawdown
name,,,,,,
markowitz_max_sharpe,261,0.389477,0.373806,0.411405,0.978172,-0.337167
equal_weight,261,0.336163,0.322877,0.405584,0.893038,-0.342852
naive_risk_parity,261,0.298146,0.286518,0.406733,0.823022,-0.346961
hrp_style,261,0.202089,0.194483,0.430932,0.627634,-0.370733
hrp_recursive,261,0.198193,0.190746,0.430007,0.620762,-0.371075


In [9]:
markowitz_window_summary = Backtester(universe, "markowitz_max_sharpe").summarize_window(
    result=backtest_results["markowitz_max_sharpe"],
    start_date="2020-03-01",
    end_date="2020-12-31",
)

pd.Series(markowitz_window_summary, name="markowitz_window_summary")


n_periods                                219
total_return                        0.486401
annualized_return                   0.577881
annualized_volatility               0.433549
sharpe_ratio                        1.269468
max_drawdown                       -0.271946
window_start             2020-03-02 00:00:00
window_end               2020-12-31 00:00:00
Name: markowitz_window_summary, dtype: object

In [10]:
mc_horizon = len(universe.get_returns_window(BACKTEST_START, BACKTEST_END))
mc_results = MonteCarloEngine.run_all(
    universe,
    horizon=mc_horizon,
    n_simulations=MC_SIMULATIONS,
    initial_value=MC_INITIAL_VALUE,
    seed=MC_SEED,
    attach=True,
)

mc_summary = pd.DataFrame(
    [
        {"name": name, **result.summary_metrics}
        for name, result in mc_results.items()
    ]
).set_index("name")

display(pd.Series({"mc_horizon": mc_horizon, "n_simulations": MC_SIMULATIONS, "seed": MC_SEED}, name="mc_config"))
display(mc_summary.sort_values("mean_terminal_value", ascending=False))


mc_horizon       261
n_simulations    300
seed              42
Name: mc_config, dtype: int64

,mean_terminal_value,median_terminal_value,std_terminal_value,min_terminal_value,max_terminal_value,prob_loss,mean_terminal_return
name,,,,,,,
markowitz_max_sharpe,1.682656,1.658032,0.287791,1.010307,2.691257,0.000000,0.682656
equal_weight,1.612333,1.578606,0.291353,0.996464,2.505523,0.003333,0.612333
naive_risk_parity,1.572204,1.566637,0.257987,0.996081,2.630626,0.003333,0.572204
hrp_style,1.561476,1.568079,0.242443,0.928057,2.396482,0.010000,0.561476
hrp_recursive,1.556499,1.533862,0.263732,0.918266,2.421578,0.006667,0.556499


## Plot With `PortfolioVisualizer`

The visualizer is the cleaned plotting layer. The next cell renders the main cleaned plot families for this four-asset demo: construction plots, the Markowitz efficient frontier, backtests, drawdowns, Monte Carlo, universe heatmaps, and HRP hierarchy diagnostics.


In [11]:
import importlib
from IPython.display import Markdown
import portafolios.plots.visualizer as visualizer_module

importlib.reload(visualizer_module)
PortfolioVisualizer = visualizer_module.PortfolioVisualizer

required_methods = [
    "plot_backtest_comparison",
    "plot_weights_bar",
    "plot_weights_pie",
    "plot_weights_scatter",
    "plot_weights_bubble",
    "plot_efficient_frontier",
    "plot_backtest",
    "plot_drawdown",
    "plot_mc_distribution",
    "plot_mc_paths",
    "plot_correlation_heatmap",
    "plot_hrp_dendrogram",
    "plot_hrp_distance_matrix",
    "plot_hrp_distance_histogram",
]
missing = [name for name in required_methods if not hasattr(PortfolioVisualizer, name)]
if missing:
    raise RuntimeError(
        "The notebook is using a stale PortfolioVisualizer definition. "
        "Restart the kernel and run all cells. Missing methods: " + ", ".join(missing)
    )

visualizer = PortfolioVisualizer(universe)
construction_names = universe.list_constructions()

display(Markdown("### Backtest Comparison"))
display(visualizer.plot_backtest_comparison())

for construction_name in construction_names:
    display(Markdown(f"### {construction_name}"))
    display(visualizer.plot_weights_bar(construction_name))
    display(visualizer.plot_weights_pie(construction_name))
    display(visualizer.plot_weights_scatter(construction_name))
    display(visualizer.plot_weights_bubble(construction_name))
    if "markowitz" in construction_name.lower():
        display(visualizer.plot_efficient_frontier(construction_name))
    display(visualizer.plot_backtest(construction_name))
    display(visualizer.plot_drawdown(construction_name))
    display(visualizer.plot_mc_distribution(construction_name))
    display(visualizer.plot_mc_paths(construction_name, max_paths=MAX_MC_PATHS_TO_SAVE))

display(Markdown("### Universe Heatmaps"))
display(visualizer.plot_correlation_heatmap(kind="correlation"))
display(visualizer.plot_correlation_heatmap(kind="covariance"))

hrp_constructions = [
    name for name in construction_names
    if universe.get_construction(name).hrp_diagnostics is not None
]

display(Markdown("### HRP Diagnostics"))
for construction_name in hrp_constructions:
    diagnostics = universe.get_construction(construction_name).hrp_diagnostics
    display(Markdown(f"#### {construction_name}"))
    if diagnostics is not None and diagnostics.linkage_matrix is not None:
        display(visualizer.plot_hrp_dendrogram(construction_name))
    display(visualizer.plot_hrp_distance_matrix(construction_name))
    display(visualizer.plot_hrp_distance_histogram(construction_name))


### Backtest Comparison

### equal_weight

### markowitz_max_sharpe

### naive_risk_parity

### hrp_style

### hrp_recursive

### Universe Heatmaps

### HRP Diagnostics

#### hrp_style

#### hrp_recursive

## Save The Standard Library Output Structure

This writes the reusable framework layout under `outputs/runs/<universe_name>/`. In the cleaned structure, numeric artifacts stay in `data/`, `constructions/`, `backtests/`, and `monte_carlo/`, while HTML figures live under `plots/`.


In [12]:
saved = visualizer.save_everything(max_mc_paths=MAX_MC_PATHS_TO_SAVE)

saved_summary = pd.Series(
    {
        "market_data_dir": str(saved["market_data_dir"].relative_to(PROJECT_ROOT)),
        "n_constructions_saved": len(saved["constructions"]),
        "n_backtests_saved": len(saved["backtests"]),
        "n_monte_carlo_saved": len(saved["monte_carlo"]),
        "construction_plot_groups": list(saved["construction_plots"].keys()),
        "backtest_plot_groups": list(saved["backtest_plots"].keys()),
        "monte_carlo_plot_groups": list(saved["monte_carlo_plots"].keys()),
    },
    name="saved_summary",
)

display(saved_summary)


market_data_dir             outputs\runs\djia_four_asset_library_walkthrou...
n_constructions_saved                                                       5
n_backtests_saved                                                           5
n_monte_carlo_saved                                                         5
construction_plot_groups    [equal_weight, markowitz_max_sharpe, naive_ris...
backtest_plot_groups        [equal_weight, markowitz_max_sharpe, naive_ris...
monte_carlo_plot_groups     [equal_weight, markowitz_max_sharpe, naive_ris...
Name: saved_summary, dtype: object

In [13]:
example_structure = {
    "data": sorted(str(path.relative_to(run_dir)) for path in (run_dir / "data").glob("*")),
    "construction_example": sorted(
        str(path.relative_to(run_dir))
        for path in (run_dir / "constructions" / "markowitz_max_sharpe").glob("*")
    ),
    "backtest_example": sorted(
        str(path.relative_to(run_dir))
        for path in (run_dir / "backtests" / "markowitz_max_sharpe").glob("*")
    ),
    "monte_carlo_example": sorted(
        str(path.relative_to(run_dir))
        for path in (run_dir / "monte_carlo" / "markowitz_max_sharpe").glob("*")
    ),
    "comparison_data": sorted(
        str(path.relative_to(run_dir))
        for path in (run_dir / "monte_carlo").glob("comparison_*.csv")
    ),
    "plot_examples": sorted(
        str(path.relative_to(run_dir))
        for path in (run_dir / "plots").rglob("*.html")
    )[:12],
}

print(json.dumps(example_structure, indent=2))


{
  "data": [
    "data\\metadata.json",
    "data\\prices.csv",
    "data\\returns.csv",
    "data\\tickers.json"
  ],
  "construction_example": [
    "constructions\\markowitz_max_sharpe\\construction_window.json",
    "constructions\\markowitz_max_sharpe\\metrics_insample.json",
    "constructions\\markowitz_max_sharpe\\params.json",
    "constructions\\markowitz_max_sharpe\\selected_assets.json",
    "constructions\\markowitz_max_sharpe\\summary.json",
    "constructions\\markowitz_max_sharpe\\weights.csv"
  ],
  "backtest_example": [
    "backtests\\markowitz_max_sharpe\\backtest_window.json",
    "backtests\\markowitz_max_sharpe\\cumulative_returns.csv",
    "backtests\\markowitz_max_sharpe\\drawdown_series.csv",
    "backtests\\markowitz_max_sharpe\\portfolio_returns.csv",
    "backtests\\markowitz_max_sharpe\\summary_metrics.json"
  ],
  "monte_carlo_example": [
    "monte_carlo\\markowitz_max_sharpe\\simulated_paths.csv",
    "monte_carlo\\markowitz_max_sharpe\\simulation_conf

## Next Step

This notebook is a compact reference for the cleaned library workflow. For the larger thesis-scale experiment grid, use [scripts/run_final_experimental_setup.py](../../../scripts/run_final_experimental_setup.py).
